In [0]:
# ====================================================================
# GOLD LAYER - REVENUE SUMMARY
# ====================================================================
# Purpose: Create daily/monthly revenue aggregations by category, 
#          region, and payment type for executive dashboards
# ====================================================================

from pyspark.sql.functions import (
    col, sum, avg, count, countDistinct, 
    round, current_timestamp, year, month, dayofmonth, date_trunc
)

# Configuration
PROJECT_NAME = "retail"
CATALOG = "workspace"
SILVER_SCHEMA = f"{PROJECT_NAME}_silver"
GOLD_SCHEMA = f"{PROJECT_NAME}_gold"

print("=" * 80)
print("💰 GOLD LAYER - REVENUE SUMMARY")
print("=" * 80)
print(f"Source: {CATALOG}.{SILVER_SCHEMA}.silver_orders_enriched")
print(f"Target: {CATALOG}.{GOLD_SCHEMA}.gold_revenue_summary")
print("=" * 80 + "\n")

In [0]:
%sql
-- Create gold schema if it doesn't exist
CREATE SCHEMA IF NOT EXISTS workspace.retail_gold
  COMMENT 'Business-ready aggregated analytics layer';

-- Verify
SHOW SCHEMAS IN workspace LIKE 'retail*';

In [0]:
# Load enriched orders
enriched_orders = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.silver_orders_enriched")

print(f"✅ Loaded enriched orders: {enriched_orders.count():,} rows")
print("=" * 80 + "\n")

In [0]:
# ====================================================================
# DAILY REVENUE AGGREGATION
# ====================================================================
print("📅 Creating daily revenue summary...\n")

daily_revenue = (enriched_orders
    .filter(col("order_status") == "delivered")  # Only completed orders
    .withColumn("order_date", date_trunc("day", col("order_purchase_timestamp")))
    .groupBy(
        "order_date",
        year("order_date").alias("year"),
        month("order_date").alias("month"),
        dayofmonth("order_date").alias("day"),
        "customer_state",
        "product_category_english",
        "primary_payment_type"
    )
    .agg(
        # Revenue metrics
        sum("item_total_value").alias("total_revenue"),
        sum("price").alias("total_product_revenue"),
        sum("freight_value").alias("total_freight_revenue"),
        
        # Order metrics
        countDistinct("order_id").alias("total_orders"),
        count("*").alias("total_items_sold"),
        countDistinct("customer_id").alias("unique_customers"),
        
        # Average metrics
        avg("item_total_value").alias("avg_item_value"),
        avg("total_payment_value").alias("avg_order_value"),
        
        # Review metrics
        avg("review_score").alias("avg_review_score"),
        
        # Delivery metrics
        avg("actual_delivery_days").alias("avg_delivery_days"),
        sum(when(col("is_late_delivery") == True, 1).otherwise(0)).alias("late_deliveries")
    )
    .withColumn(
        "late_delivery_rate",
        round((col("late_deliveries") / col("total_orders")) * 100, 2)
    )
    .withColumn("_created_at", current_timestamp())
)

print(f"✅ Daily revenue records: {daily_revenue.count():,}")
print("=" * 80 + "\n")

In [0]:
# ====================================================================
# MONTHLY REVENUE AGGREGATION
# ====================================================================
print("📆 Creating monthly revenue summary...\n")

monthly_revenue = (enriched_orders
    .filter(col("order_status") == "delivered")
    .withColumn("order_month", date_trunc("month", col("order_purchase_timestamp")))
    .groupBy(
        "order_month",
        year("order_month").alias("year"),
        month("order_month").alias("month"),
        "customer_state",
        "product_category_english"
    )
    .agg(
        # Revenue metrics
        sum("item_total_value").alias("total_revenue"),
        sum("price").alias("total_product_revenue"),
        sum("freight_value").alias("total_freight_revenue"),
        
        # Order metrics
        countDistinct("order_id").alias("total_orders"),
        count("*").alias("total_items_sold"),
        countDistinct("customer_id").alias("unique_customers"),
        countDistinct("product_id").alias("unique_products_sold"),
        
        # Average metrics
        avg("item_total_value").alias("avg_item_value"),
        avg("total_payment_value").alias("avg_order_value"),
        avg("review_score").alias("avg_review_score"),
        
        # Delivery metrics
        avg("actual_delivery_days").alias("avg_delivery_days"),
        sum(when(col("is_late_delivery") == True, 1).otherwise(0)).alias("late_deliveries")
    )
    .withColumn(
        "late_delivery_rate",
        round((col("late_deliveries") / col("total_orders")) * 100, 2)
    )
    .withColumn("_created_at", current_timestamp())
)

print(f"✅ Monthly revenue records: {monthly_revenue.count():,}")
print("=" * 80 + "\n")

In [0]:
# Write daily revenue summary
daily_table_name = f"{CATALOG}.{GOLD_SCHEMA}.gold_revenue_daily"

(daily_revenue.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(daily_table_name))

print(f"✅ Written: {daily_table_name}")
print(f"   Rows: {daily_revenue.count():,}")
print(f"   Columns: {len(daily_revenue.columns)}")
print("=" * 80 + "\n")

In [0]:
# Write monthly revenue summary
monthly_table_name = f"{CATALOG}.{GOLD_SCHEMA}.gold_revenue_monthly"

(monthly_revenue.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(monthly_table_name))

print(f"✅ Written: {monthly_table_name}")
print(f"   Rows: {monthly_revenue.count():,}")
print(f"   Columns: {len(monthly_revenue.columns)}")
print("=" * 80 + "\n")

In [0]:
# ====================================================================
# OVERALL SUMMARY STATISTICS
# ====================================================================
print("📊 Creating overall revenue statistics...\n")

overall_stats = (enriched_orders
    .filter(col("order_status") == "delivered")
    .agg(
        sum("item_total_value").alias("total_revenue"),
        countDistinct("order_id").alias("total_orders"),
        countDistinct("customer_id").alias("total_customers"),
        countDistinct("product_id").alias("total_products_sold"),
        avg("item_total_value").alias("avg_item_value"),
        avg("review_score").alias("avg_review_score")
    )
    .withColumn("_created_at", current_timestamp())
)

overall_table_name = f"{CATALOG}.{GOLD_SCHEMA}.gold_revenue_overall"

(overall_stats.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(overall_table_name))

print(f"✅ Written: {overall_table_name}")
print("=" * 80 + "\n")

In [0]:
%sql
-- Top 10 revenue days
SELECT 
    order_date,
    SUM(total_revenue) as daily_revenue,
    SUM(total_orders) as orders,
    SUM(unique_customers) as customers
FROM workspace.retail_gold.gold_revenue_daily
GROUP BY order_date
ORDER BY daily_revenue DESC
LIMIT 10;

In [0]:
%sql
-- Monthly revenue trend
SELECT 
    order_month,
    SUM(total_revenue) as monthly_revenue,
    SUM(total_orders) as orders,
    SUM(unique_customers) as customers,
    ROUND(AVG(avg_review_score), 2) as avg_review
FROM workspace.retail_gold.gold_revenue_monthly
GROUP BY order_month
ORDER BY order_month;

In [0]:
%sql
-- Top product categories
SELECT 
    product_category_english,
    SUM(total_revenue) as category_revenue,
    SUM(total_items_sold) as items_sold,
    ROUND(AVG(avg_review_score), 2) as avg_review
FROM workspace.retail_gold.gold_revenue_monthly
GROUP BY product_category_english
ORDER BY category_revenue DESC
LIMIT 15;

In [0]:
%sql
-- Top states by revenue
SELECT 
    customer_state,
    SUM(total_revenue) as state_revenue,
    SUM(total_orders) as orders,
    SUM(unique_customers) as customers
FROM workspace.retail_gold.gold_revenue_monthly
GROUP BY customer_state
ORDER BY state_revenue DESC
LIMIT 10;

In [0]:
print("\n" + "=" * 80)
print("✅ REVENUE SUMMARY COMPLETE")
print("=" * 80)
print("\nCreated tables:")
print(f"  - {CATALOG}.{GOLD_SCHEMA}.gold_revenue_daily")
print(f"  - {CATALOG}.{GOLD_SCHEMA}.gold_revenue_monthly")
print(f"  - {CATALOG}.{GOLD_SCHEMA}.gold_revenue_overall")
print("\n📝 Next: Run 02_customer_cohorts.py")
print("=" * 80 + "\n")